# Predict Chemical Reaction

## Datasets
| Dataset                            | Main Purpose                                                                | Data Type                                                                                     |                                            Approx. Size | Contains Mechanism?           | Contains Conditions?                                   | Best Use for Your Project                                                        | Difficulty      |
| ---------------------------------- | --------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------- | ------------------------------------------------------: | ----------------------------- | ------------------------------------------------------ | -------------------------------------------------------------------------------- | --------------- |
| **USPTO / USPTO-50K / USPTO-Full** | Forward reaction prediction, retrosynthesis benchmark                       | Reaction SMILES from patent reactions                                                         | USPTO-50K: about 50k reactions; USPTO-Full: much larger | Usually **No**                | Usually limited / inconsistent                         | **Best starting dataset** for `reactant graph → product SMILES`                  | **Easy–Medium** |
| **ORD / Open Reaction Database**   | Structured real reaction data for ML, synthesis planning, experiment design | Rich reaction records, often protobuf/schema-based                                            |          Large and growing; public ORD data repo exists | Usually **No full mechanism** | **Yes**, often more structured than USPTO              | Good for later: reaction condition prediction, yield/condition-aware models      | **Medium–Hard** |
| **PMechDB**                        | Elementary **polar** reaction mechanism modeling                            | Canonicalized, balanced elementary polar steps with atom mapping and arrow-pushing mechanisms |                              Mechanism-focused database | **Yes**                       | Less about lab conditions; more about elementary steps | Good for advanced mechanistic model / reaction center / electron-flow prediction | **Hard**        |
| **RMechDB**                        | Elementary **radical** reaction mechanism modeling                          | Elementary radical steps                                                                      |                  Around 5,300+ radical elementary steps | **Yes**                       | Mostly mechanism-focused                               | Good for radical-specific mechanism prediction, not general product prediction   | **Hard**        |


## Pretriained GNN model list
| Model                         | Type                                                     | Main Use                                                 | Fit for Your Project                 |
| ----------------------------- | -------------------------------------------------------- | -------------------------------------------------------- | ------------------------------------ |
| **GROVER**                    | Pretrained molecular graph Transformer / GNN-style model | Molecular representation, property prediction            | **Good**                             |
| **GraphMVP**                  | 2D graph + 3D geometry pretrained model                  | Molecular representation with 3D structural knowledge    | **Good, but more complex**           |
| **Chemprop MPNN**             | Message Passing Neural Network                           | Molecular property prediction, molecular embeddings      | **Practical and easier to start**    |
| **TorchDrug pretrained GNNs** | GIN/GCN/GNN pretraining framework                        | Experimenting with molecular pretraining and fine-tuning | **Good for research prototypes**     |
| **DGL-LifeSci Model Zoo**     | Collection of chemistry GNN models                       | Molecular property prediction, graph learning baselines  | **Good reference / baseline source** |


## Pretrained Transformer model list (Decoder)
| Model                     | Type                                        | Main Use                                                              | Fit for Your Project                           |
| ------------------------- | ------------------------------------------- | --------------------------------------------------------------------- | ---------------------------------------------- |
| **Chemformer**            | BART-style molecular seq2seq Transformer    | Reaction prediction, retrosynthesis, molecular optimization           | **Best fit**                                   |
| **MolT5**                 | T5-style molecular/text seq2seq Transformer | SMILES generation, molecule captioning, text-to-molecule tasks        | **Good, flexible**                             |
| **Molecular Transformer** | Transformer seq2seq for reactions           | Forward reaction prediction                                           | **Strong baseline / classic reference**        |
| **RXNMapper / RXNFP**     | BERT-style reaction Transformer             | Reaction fingerprinting, atom mapping support, reaction understanding | **Good auxiliary model, not ideal as decoder** |
| **ChemBERTa**             | BERT-style molecular language model         | Molecular representation, property prediction                         | **Not good as decoder; encoder-only**          |
| **MegaMolBART**           | BART-style molecular generation model       | Molecular representation and SMILES generation                        | **Good, but heavier**                          |
| **MolBART**               | BART-style molecular Transformer            | SMILES denoising, molecule generation                                 | **Good alternative to Chemformer**             |
| **T5-small / BART-base**  | General pretrained seq2seq Transformer      | Generic text generation, customizable decoder                         | **Usable, but not chemistry-specialized**      |


## SMILES -> Graph

In [95]:
%pip install rdkit
%pip install torch_geometric
%pip install transformers sentencepiece tensorboard

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [96]:
from rdkit import Chem
from rdkit.Chem import PandasTools
import pandas as pd
from rdkit import Chem
import torch
from torch_geometric.data import Data
import re
import os
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader
import torch.nn as nn
from torch_geometric.nn import GINEConv
from torch_geometric.utils import to_dense_batch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
from torch.utils.tensorboard import SummaryWriter

In [97]:
def smiles_to_mol(smiles):
    """
    Convert a SMILES string into an RDKit Mol object.
    Returns None if the SMILES string is invalid.
    """
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"Invalid SMILES: {smiles}")
        return None

    return mol


def print_mol_basic_info(smiles):
    """
    Print basic RDKit molecule information.
    """
    mol = smiles_to_mol(smiles)

    if mol is None:
        return

    canonical_smiles = Chem.MolToSmiles(mol, canonical=True)

    print("Original SMILES:", smiles)
    print("Canonical SMILES:", canonical_smiles)
    print("Number of atoms:", mol.GetNumAtoms())
    print("Number of bonds:", mol.GetNumBonds())

    return mol

In [98]:
reactant_1 = "CCO"    # ethanol
reactant_2 = "O=O"    # oxygen

mol_1 = print_mol_basic_info(reactant_1)
print()
mol_2 = print_mol_basic_info(reactant_2)

Original SMILES: CCO
Canonical SMILES: CCO
Number of atoms: 3
Number of bonds: 2

Original SMILES: O=O
Canonical SMILES: O=O
Number of atoms: 2
Number of bonds: 1


## Data Stucture
- Atoms
- Edges_index (Atoms - Atoms)
- Edges_attr - bond feature matrix

In [99]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic())
    ]


def bond_features(bond):
    bond_type = bond.GetBondType()

    return [
        int(bond_type == Chem.rdchem.BondType.SINGLE),
        int(bond_type == Chem.rdchem.BondType.DOUBLE),
        int(bond_type == Chem.rdchem.BondType.TRIPLE),
        int(bond_type == Chem.rdchem.BondType.AROMATIC),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing())
    ]


def smiles_to_pyg_data(reactant_smiles):
    mol = Chem.MolFromSmiles(reactant_smiles)

    if mol is None:
        raise ValueError(f"Invalid reactant SMILES: {reactant_smiles}")

    x = []
    for atom in mol.GetAtoms():
        x.append(atom_features(atom))

    x = torch.tensor(x, dtype=torch.float)

    edge_index = []
    edge_attr = []

    for bond in mol.GetBonds():
        start = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()

        bond_feat = bond_features(bond)

        edge_index.append([start, end])
        edge_index.append([end, start])

        edge_attr.append(bond_feat)
        edge_attr.append(bond_feat)

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 6), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    return Data(
        x=x,
        edge_index=edge_index,  
        edge_attr=edge_attr
    )

In [100]:
data = smiles_to_pyg_data("CCO")

print(data)
print(data.x)
print(data.edge_index)
print(data.edge_attr)

Data(x=[3, 4], edge_index=[2, 4], edge_attr=[4, 6])
tensor([[6., 1., 0., 0.],
        [6., 2., 0., 0.],
        [8., 1., 0., 0.]])
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.]])


### Data Example
```
Data(x=[3, 4], edge_index=[2, 4], edge_attr=[4, 6])
tensor([[6., 1., 0., 0.],
        [6., 2., 0., 0.],
        [8., 1., 0., 0.]])
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.]])
```
x = [3 , 4] -> 3 atoms\
4 => Atom Feature

```
atom_featue = [
        atomic_number, -> Atom number
        degree, -> how many atoms are connected
        formal_charge, -> Plus or negative ion
        is_aromatic -> Ring or not
]
```
edge_index=[2, 4] -> [2, number of edges]\

|From|To|Meaning|
|---|---|---|
|0|1| c0 $\rightarrow$ c1|
|1|0| c1 $\rightarrow$ c0|
|1|2| c1 $\rightarrow$ c2|
|2|1| c2 $\rightarrow$ c1|

 edge_attr=[4, 6] => [number of edges, 6]\

```
bond_feature = [
    is_single,
    is_double,
    is_triple,
    is_aromatic,
    is_conjugated,
    is_in_ring
]
```

```
tensor([[1., 0., 0., 0., 0., 0.], -> is_single
        [1., 0., 0., 0., 0., 0.], -> is_single
        [1., 0., 0., 0., 0., 0.], -> is_single
        [1., 0., 0., 0., 0., 0.]]) -> is_single
```

## GNN DataSets

### USPTO-50K

- [link](https://figshare.com/articles/dataset/USPTO-50K_raw_/25459573?file=45206101)

In [101]:
def remove_atom_mapping(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)

    return Chem.MolToSmiles(mol, canonical=True)


def parse_uspto_reaction(rxn_smiles):
    parts = rxn_smiles.split(">")

    if len(parts) != 3:
        return None

    reactants, reagents, products = parts

    # reactants + reagents를 input graph로 사용
    if reagents.strip():
        input_smiles = reactants + "." + reagents
    else:
        input_smiles = reactants

    input_smiles = remove_atom_mapping(input_smiles)
    product_smiles = remove_atom_mapping(products)

    if input_smiles is None or product_smiles is None:
        return None

    return {
        "reactants": input_smiles,
        "product": product_smiles
    }


def load_uspto_csv(csv_path, limit=None):
    df = pd.read_csv(csv_path)

    reaction_data = []

    for rxn in df["reactants>reagents>production"]:
        parsed = parse_uspto_reaction(rxn)

        if parsed is not None:
            reaction_data.append(parsed)

        if limit is not None and len(reaction_data) >= limit:
            break

    return reaction_data

In [102]:
path = "uspto50k"

train_reactions = load_uspto_csv(f"{path}/raw_train.csv", limit=1000)
val_reactions = load_uspto_csv(f"{path}/raw_val.csv", limit=200)
test_reactions = load_uspto_csv(f"{path}/raw_test.csv", limit=200)

In [103]:
'''
reaction_data = [
    {"reactants": "CCO.O=C=O", "product": "CC(=O)O"},
    {"reactants": "CCBr.O", "product": "CCO"},
    {"reactants": "CCN.O=C=O", "product": "CCNC(=O)O"},
]
'''

'\nreaction_data = [\n    {"reactants": "CCO.O=C=O", "product": "CC(=O)O"},\n    {"reactants": "CCBr.O", "product": "CCO"},\n    {"reactants": "CCN.O=C=O", "product": "CCNC(=O)O"},\n]\n'

In [104]:
class ReactionDataset(Dataset):
    def __init__(self, reactions, tokenizer, max_len=256):
        self.reactions = reactions
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.reactions)

    def __getitem__(self, idx):
        item = self.reactions[idx]

        reactants = item["reactants"]
        product = item["product"]

        # Reactants -> PyG graph
        data = smiles_to_pyg_data(reactants)

        # Product SMILES -> MolT5 token ids
        encoded = self.tokenizer(
            product,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        data.product_ids = encoded["input_ids"]
        data.product_attention_mask = encoded["attention_mask"]

        return data

## GNN Encoder

In [105]:
atom_dim = 4
bond_dim = 6

In [106]:
class GNNEncoder(nn.Module):
    def __init__(self, atom_dim=4, bond_dim=6, hidden_dim=256, num_layers=4):
        super().__init__()

        self.atom_proj = nn.Linear(atom_dim, hidden_dim)
        self.bond_proj = nn.Linear(bond_dim, hidden_dim)

        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )

            self.layers.append(GINEConv(mlp))
            self.norms.append(nn.LayerNorm(hidden_dim))

    def forward(self, x, edge_index, edge_attr):
        h = self.atom_proj(x)

        if edge_attr.numel() > 0:
            edge_attr = self.bond_proj(edge_attr)
        else:
            edge_attr = edge_attr.new_empty((0, h.size(-1)))

        for conv, norm in zip(self.layers, self.norms):
            h_new = conv(h, edge_index, edge_attr)
            h = norm(h + h_new)

        return h

## MODEL

- First test = MOLT5-small
- Fainl = Chemformer

In [107]:
class GNNMolT5ReactionModel(nn.Module):
    def __init__(
        self,
        gnn_encoder,
        molt5_model,
        tokenizer,
        gnn_hidden_dim=256,
        freeze_molt5=False
    ):
        super().__init__()

        self.gnn = gnn_encoder
        self.molt5 = molt5_model
        self.tokenizer = tokenizer

        # T5/MolT5 hidden dimension
        transformer_hidden_dim = self.molt5.config.d_model

        # GNN hidden dim -> MolT5 hidden dim
        self.adapter = nn.Linear(
            gnn_hidden_dim,
            transformer_hidden_dim
        )

        if freeze_molt5:
            for param in self.molt5.parameters():
                param.requires_grad = False

    def forward(self, batch):
        """
        batch should contain:
        - batch.x
        - batch.edge_index
        - batch.edge_attr
        - batch.batch
        - batch.product_ids
        """

        # 1. Encode reactant molecular graph with our GNN
        atom_embeddings = self.gnn(
            batch.x,
            batch.edge_index,
            batch.edge_attr
        )
        # atom_embeddings: [total_atoms, gnn_hidden_dim]

        # 2. Convert PyG sparse batch to dense batch for Transformer memory
        graph_memory, graph_mask = to_dense_batch(
            atom_embeddings,
            batch.batch
        )
        # graph_memory: [B, max_atoms, gnn_hidden_dim]
        # graph_mask:   [B, max_atoms], True = real atom, False = padding

        # 3. Project GNN memory to MolT5 hidden dimension
        graph_memory = self.adapter(graph_memory)
        # graph_memory: [B, max_atoms, molt5_hidden_dim]

        # 4. Wrap graph memory as if it were MolT5 encoder output
        encoder_outputs = BaseModelOutput(
            last_hidden_state=graph_memory
        )

        # 5. Prepare labels
        labels = batch.product_ids.clone()

        # Hugging Face ignores labels == -100 in loss
        labels[labels == self.tokenizer.pad_token_id] = -100

        # 6. Run MolT5 decoder using graph memory as encoder memory
        outputs = self.molt5(
            encoder_outputs=encoder_outputs,
            attention_mask=graph_mask.long(),
            labels=labels,
            return_dict=True
        )

        return outputs

### Create Model

In [108]:
model_name = "laituan245/molt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
molt5 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 192/192 [00:00<00:00, 36391.45it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [109]:
gnn_encoder = GNNEncoder(
    atom_dim=4,
    bond_dim=6,
    hidden_dim=256,
    num_layers=4
)

model = GNNMolT5ReactionModel(
    gnn_encoder=gnn_encoder,
    molt5_model=molt5,
    tokenizer=tokenizer,
    gnn_hidden_dim=256,
    freeze_molt5=False
)

## Load DATA set

### USPTO-50K

In [110]:
train_dataset = ReactionDataset(
    reactions=train_reactions,
    tokenizer=tokenizer,
    max_len=128
)

val_dataset = ReactionDataset(
    reactions=val_reactions,
    tokenizer=tokenizer,
    max_len=128
)

test_dataset = ReactionDataset(
    reactions=test_reactions,
    tokenizer=tokenizer,
    max_len=128
)

In [111]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

## Training

In [112]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 62563), started 0:03:10 ago. (Use '!kill 62563' to kill it.)

### Token Accuracy

In [113]:
def compute_token_accuracy(logits, labels):
    """
    logits: [B, T, vocab_size]
    labels: [B, T]
    labels에서 -100은 loss/accuracy 계산에서 무시
    """
    preds = torch.argmax(logits, dim=-1)

    mask = labels != -100

    correct = (preds == labels) & mask

    total_correct = correct.sum().item()
    total_tokens = mask.sum().item()

    if total_tokens == 0:
        return 0.0

    return total_correct / total_tokens

### Train one Epoch

In [114]:
def train_one_epoch(model, train_loader, optimizer, device, epoch, writer=None):
    model.train()

    total_loss = 0.0
    total_token_acc = 0.0
    num_batches = 0

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)

        outputs = model(batch)
        loss = outputs.loss

        labels = batch.product_ids.clone()
        labels[labels == model.tokenizer.pad_token_id] = -100

        token_acc = compute_token_accuracy(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()
        total_token_acc += token_acc
        num_batches += 1

        global_step = epoch * len(train_loader) + step

        if writer is not None:
            writer.add_scalar("Train/Batch_Loss", loss.item(), global_step)
            writer.add_scalar("Train/Batch_Token_Accuracy", token_acc, global_step)

        if step % 10 == 0:
            print(
                f"Epoch {epoch + 1} | "
                f"Step {step}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Token Acc: {token_acc:.4f}"
            )

    avg_loss = total_loss / num_batches
    avg_token_acc = total_token_acc / num_batches

    return avg_loss, avg_token_acc

### Evaluation

In [115]:
@torch.no_grad()
def evaluate(model, val_loader, device):
    model.eval()

    total_loss = 0.0
    total_token_acc = 0.0
    num_batches = 0

    for batch in val_loader:
        batch = batch.to(device)

        outputs = model(batch)
        loss = outputs.loss

        labels = batch.product_ids.clone()
        labels[labels == model.tokenizer.pad_token_id] = -100

        token_acc = compute_token_accuracy(outputs.logits, labels)

        total_loss += loss.item()
        total_token_acc += token_acc
        num_batches += 1

    avg_loss = total_loss / num_batches
    avg_token_acc = total_token_acc / num_batches

    return avg_loss, avg_token_acc

### Optimizer

In [116]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

optimizer = torch.optim.AdamW([
    {"params": model.gnn.parameters(), "lr": 1e-4},
    {"params": model.adapter.parameters(), "lr": 1e-4},
    {"params": model.molt5.parameters(), "lr": 1e-5},
])

### TensorBoard Writer

In [117]:
log_dir = "runs/gnn_molt5_reaction"
writer = SummaryWriter(log_dir=log_dir)

### Training Loop

In [118]:
num_epochs = 10

history = {
    "train_loss": [],
    "train_token_acc": [],
    "val_loss": [],
    "val_token_acc": []
}

best_val_loss = float("inf")

for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch + 1}/{num_epochs} =====")

    train_loss, train_token_acc = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        device=device,
        epoch=epoch,
        writer=writer
    )

    val_loss, val_token_acc = evaluate(
        model=model,
        val_loader=val_loader,
        device=device
    )

    history["train_loss"].append(train_loss)
    history["train_token_acc"].append(train_token_acc)
    history["val_loss"].append(val_loss)
    history["val_token_acc"].append(val_token_acc)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Token Acc: {train_token_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Token Acc: {val_token_acc:.4f}"
    )

    writer.add_scalar("Epoch/Train_Loss", train_loss, epoch)
    writer.add_scalar("Epoch/Train_Token_Accuracy", train_token_acc, epoch)
    writer.add_scalar("Epoch/Val_Loss", val_loss, epoch)
    writer.add_scalar("Epoch/Val_Token_Accuracy", val_token_acc, epoch)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        os.makedirs("checkpoints", exist_ok=True)

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_token_acc": val_token_acc,
            },
            "checkpoints/best_gnn_molt5_model.pt"
        )

        print("Saved best model.")

writer.close()


===== Epoch 1/10 =====
Epoch 1 | Step 0/500 | Loss: 21.7303 | Token Acc: 0.0000
Epoch 1 | Step 10/500 | Loss: 5.0334 | Token Acc: 0.1184
Epoch 1 | Step 20/500 | Loss: 3.3386 | Token Acc: 0.1509
Epoch 1 | Step 30/500 | Loss: 3.0684 | Token Acc: 0.3421
Epoch 1 | Step 40/500 | Loss: 3.7273 | Token Acc: 0.1263
Epoch 1 | Step 50/500 | Loss: 3.3692 | Token Acc: 0.2718
Epoch 1 | Step 60/500 | Loss: 3.6900 | Token Acc: 0.1111
Epoch 1 | Step 70/500 | Loss: 3.4563 | Token Acc: 0.1515
Epoch 1 | Step 80/500 | Loss: 3.2301 | Token Acc: 0.2029
Epoch 1 | Step 90/500 | Loss: 3.3510 | Token Acc: 0.1781
Epoch 1 | Step 100/500 | Loss: 3.3478 | Token Acc: 0.2449
Epoch 1 | Step 110/500 | Loss: 3.6175 | Token Acc: 0.0472
Epoch 1 | Step 120/500 | Loss: 3.2415 | Token Acc: 0.2283
Epoch 1 | Step 130/500 | Loss: 3.1648 | Token Acc: 0.1387
Epoch 1 | Step 140/500 | Loss: 3.1311 | Token Acc: 0.3065
Epoch 1 | Step 150/500 | Loss: 2.8820 | Token Acc: 0.3256
Epoch 1 | Step 160/500 | Loss: 3.4848 | Token Acc: 0.2396
